<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/uni_mol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit
!pip install unimol_tools
!pip install unimol-tools huggingface_hub --upgrade
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.9/120.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 40.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 31.8 MB/s eta 0:00:00


In [2]:
import os
import torch
import numpy as np
from rdkit import Chem
from unimol_tools import UniMolRepr
import warnings
from sklearn.metrics import roc_auc_score, matthews_corrcoef, average_precision_score, f1_score
warnings.filterwarnings('ignore')

SDF_BASE_DIR = "/content/CV_Folds"
OUTPUT_DIR = "./data/embeddings"
FOLDS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Loading Uni-Mol...")
unimol = UniMolRepr(data_type='molecule', remove_hs=False, model_name='unimolv2')

def process_fold_files(active_sdf_path, negative_sdf_path, output_name):
    """Reads both active and negative SDFs, combines them, and generates Uni-Mol embeddings."""

    # Initialize as a dictionary of lists
    custom_data = {
        "atoms": [],
        "coordinates": []
    }
    labels = []

    # Helper function to read an SDF and assign a fixed label based on the file source
    def read_sdf(sdf_path, assigned_label):
        if not os.path.exists(sdf_path):
            print(f"Warning: File not found: {sdf_path}")
            return

        supplier = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=True)
        for mol in supplier:
            if mol is None:
                continue

            labels.append(assigned_label)
            conf = mol.GetConformer()
            atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
            coords = conf.GetPositions().tolist()

            # Append to the respective lists in the dictionary
            custom_data["atoms"].append(atoms)
            custom_data["coordinates"].append(coords)

    # Read actives (Label 1.0) and negatives (Label 0.0)
    read_sdf(active_sdf_path, 1.0)
    read_sdf(negative_sdf_path, 0.0)

    if len(labels) == 0:
        print(f"No valid molecules found for {output_name}. Skipping.")
        return

    print(f"Extracting representations for {output_name} ({len(labels)} compounds)...")

  # Get representations
    repr_output = unimol.get_repr(custom_data, return_atomic_reprs=False)


    # This check extracts the [CLS] embeddings regardless of the structure.
    if isinstance(repr_output, dict):
        # Case 1: It returned a dictionary
        cls_embeddings = np.array(repr_output['cls_repr'])

    elif isinstance(repr_output, list) or isinstance(repr_output, np.ndarray):
        if len(repr_output) > 0 and isinstance(repr_output[0], dict):
            # Case 2: It returned a list of dictionaries
            cls_embeddings = np.array([item['cls_repr'] for item in repr_output])
        else:
            # Case 3: It returned the list of embeddings directly
            cls_embeddings = np.array(repr_output)

    else:
        raise ValueError(f"Unexpected output type from Uni-Mol: {type(repr_output)}")

    # Check shape to ensure it matches (Num_Molecules, Hidden_Dimension)
    print(f"Extracted embeddings shape: {cls_embeddings.shape}")

    # Convert to PyTorch tensors
    tensor_embeddings = torch.tensor(cls_embeddings, dtype=torch.float32)
    tensor_labels = torch.tensor(labels, dtype=torch.float32)

    # Save the combined tensor file
    out_file = os.path.join(OUTPUT_DIR, f"{output_name}.pt")
    torch.save({"embeddings": tensor_embeddings, "labels": tensor_labels}, out_file)
    print(f"Saved {tensor_embeddings.shape[0]} compounds to {out_file}\n")

if __name__ == "__main__":
    for i in range(1, FOLDS + 1):
        print(f"Processing Fold {i}:")

        # Define paths based on your specific naming convention
        train_actives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Train_Actives_3D.sdf")
        train_negatives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Train_Negatives_3D.sdf")

        val_actives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Val_Actives_3D.sdf")
        val_negatives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Val_Negatives_3D.sdf")

        # Process and combine Train set
        process_fold_files(train_actives, train_negatives, f"Fold_{i}_Train")

        # Process and combine Val set
        process_fold_files(val_actives, val_negatives, f"Fold_{i}_Val")

print("All Uni-Mol embeddings generated successfully!")

2026-07-17 21:18:16 | unimol_tools/weights/weighthub.py | 54 | INFO | Uni-Mol Tools | Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
INFO:Uni-Mol Tools:Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
2026-07-17 21:18:16 | unimol_tools/weights/weighthub.py | 95 | INFO | Uni-Mol Tools | Downloading modelzoo/84M/checkpoint.pt
INFO:Uni-Mol Tools:Downloading modelzoo/84M/checkpoint.pt


Loading Uni-Mol...


DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=3 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7a3c8b157830>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7a3c8b467ed0> server_hostname='huggingface.co' timeout=3
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7a3c8b68fa40>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json; charset=utf-8'), (b'T

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

DEBUG:filelock:Attempting to acquire lock 134400449882896 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock


...

Extracted embeddings shape: (135, 768)
Saved 135 compounds to ./data/embeddings/Fold_4_Val.pt

Processing Fold 5:
Extracting representations for Fold_5_Train (550 compounds)...


2026-07-17 21:18:55 | unimol_tools/tasks/trainer.py | 78 | INFO | Uni-Mol Tools | Number of GPUs available: 1
INFO:Uni-Mol Tools:Number of GPUs available: 1
2026-07-17 21:18:55 | unimol_tools/tasks/trainer.py | 98 | INFO | Uni-Mol Tools | Using single GPU.
INFO:Uni-Mol Tools:Using single GPU.
100%|██████████| 18/18 [00:04<00:00,  3.64it/s]
2026-07-17 21:19:00 | unimol_tools/tasks/trainer.py | 78 | INFO | Uni-Mol Tools | Number of GPUs available: 1
INFO:Uni-Mol Tools:Number of GPUs available: 1
2026-07-17 21:19:00 | unimol_tools/tasks/trainer.py | 98 | INFO | Uni-Mol Tools | Using single GPU.
INFO:Uni-Mol Tools:Using single GPU.


Extracted embeddings shape: (550, 768)
Saved 550 compounds to ./data/embeddings/Fold_5_Train.pt

Extracting representations for Fold_5_Val (135 compounds)...


100%|██████████| 5/5 [00:01<00:00,  4.12it/s]

Extracted embeddings shape: (135, 768)
Saved 135 compounds to ./data/embeddings/Fold_5_Val.pt

All Uni-Mol embeddings generated successfully!


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.40, gamma=2.0, smoothing=0.1):
        super(FocalLossWithSmoothing, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        # 1. Apply Label Smoothing
        targets_smoothed = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing

        # 2. Compute base BCE Loss using SMOOTHED targets
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets_smoothed, reduction='none')

        # 3. Compute pt explicitly using hard targets (Fixing the math bug)
        probs = torch.sigmoid(inputs)
        pt = probs * targets + (1.0 - probs) * (1.0 - targets)

        # 4. Construct the alpha_t term based on the original (unsmoothed) targets
        alpha_t = targets * self.alpha + (1.0 - targets) * (1.0 - self.alpha)

        # 5. Calculate Focal Loss
        focal_weight = alpha_t * (1.0 - pt) ** self.gamma

        return torch.mean(focal_weight * BCE_loss)


class UniMolMLPHead(nn.Module):
    def __init__(self, input_dim=768, hidden_dim_1=512, hidden_dim_2=128, dropout_rate=0.3):
        super(UniMolMLPHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_2, 1)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x).squeeze(-1)

In [8]:
import os
import torch
import optuna
from torch.utils.data import Dataset

class UniMolFoldDataset(Dataset):
    def __init__(self, fold_idx, is_train, base_dir="./data/embeddings"):
        phase = "Train" if is_train else "Val"
        file_path = os.path.join(base_dir, f"Fold_{fold_idx}_{phase}.pt")
        data = torch.load(file_path)
        self.embeddings = data["embeddings"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


In [9]:
from torch.utils.data import DataLoader
import torch.optim as optim
def train_fold(fold_idx, epochs=150, batch_size=32, lr=5e-4, hidden_dim_1=512, hidden_dim_2=128, dropout_rate=0.3):
    print(f"Starting Fold {fold_idx}/5")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = UniMolFoldDataset(fold_idx, is_train=True)
    val_dataset = UniMolFoldDataset(fold_idx, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = UniMolMLPHead(input_dim=768,
                          hidden_dim_1=hidden_dim_1,
                          hidden_dim_2=hidden_dim_2,
                          dropout_rate=dropout_rate).to(device)

    criterion = FocalLossWithSmoothing(alpha=0.40, gamma=2.0, smoothing=0.1)



    # Aggressive Weight Decay (1e-2) and patience (20)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=20)


    # Track Best AUC
    best_auc = 0.0
    best_metrics = {}


    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for embeddings, labels in train_loader:
            if embeddings.size(0) == 1: continue
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(embeddings)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * embeddings.size(0)


        train_loss /= len(train_loader.dataset)


        model.eval()
        val_loss = 0.0
        all_logits = []
        all_labels = []


        with torch.no_grad():
            for embeddings, labels in val_loader:
                embeddings, labels = embeddings.to(device), labels.to(device)
                logits = model(embeddings)
                loss = criterion(logits, labels)
                val_loss += loss.item() * embeddings.size(0)
                all_logits.extend(logits.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())


        val_loss /= len(val_loader.dataset)
        raw_probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
        raw_labels = np.array(all_labels)


        # Test-Time Conformer Consensus (Boltzmann Averaging)
        NUM_CONFORMERS = 5


        if len(raw_probs) % NUM_CONFORMERS == 0:
            probs = raw_probs.reshape(-1, NUM_CONFORMERS).mean(axis=1)
            all_labels = raw_labels.reshape(-1, NUM_CONFORMERS)[:, 0]
        else:
            print("\n[WARNING] Validation set not divisible by 5. Test-Time Consensus skipped!\n")
            probs = raw_probs
            all_labels = raw_labels


        # Calculate Threshold-Independent Metrics
        try:
            auc = roc_auc_score(all_labels, probs)
            pr_auc = average_precision_score(all_labels, probs)
        except ValueError:
            auc, pr_auc = 0.0, 0.0


        # Dynamic Optimal Thresholding for MCC & F1
        best_fold_mcc = -1.0
        best_fold_f1 = 0.0
        best_thresh = 0.5


        for thresh in np.arange(0.20, 0.85, 0.05):
            temp_preds = (probs > thresh).astype(int)
            try:
                temp_mcc = matthews_corrcoef(all_labels, temp_preds)
                temp_f1 = f1_score(all_labels, temp_preds)
            except ValueError:
                temp_mcc, temp_f1 = 0.0, 0.0


            if temp_mcc > best_fold_mcc:
                best_fold_mcc = temp_mcc
                best_fold_f1 = temp_f1
                best_thresh = thresh


        scheduler.step(auc)


        if auc > best_auc:
            best_auc = auc
            best_metrics = {"auc": auc, "pr_auc": pr_auc, "f1": best_fold_f1, "mcc": best_fold_mcc, "thresh": best_thresh}
            torch.save(model.state_dict(), f"best_model_fold_{fold_idx}.pth")


        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d} | Loss: {val_loss:.4f} | AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f} | F1: {best_fold_f1:.4f} | MCC: {best_fold_mcc:.4f}")


    print(f"Fold {fold_idx} Final -> AUC: {best_metrics['auc']:.4f} | PR-AUC: {best_metrics['pr_auc']:.4f} | F1: {best_metrics['f1']:.4f} | MCC: {best_metrics['mcc']:.4f}")
    return best_metrics


In [10]:
if __name__ == "__main__":

    # 1.Optuna Search
    def objective(trial):
        # Let Optuna pick the parameters
        hd_1 = trial.suggest_categorical('hidden_dim_1', [256, 512, 768])
        hd_2 = trial.suggest_categorical('hidden_dim_2', [64, 128, 256])
        drop = trial.suggest_float('dropout_rate', 0.2, 0.6)
        lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

        # Run your existing train_fold on Fold 1 for 50 epochs
        # Return the AUC so Optuna knows how well it did
        res = train_fold(fold_idx=1, epochs=50, batch_size=32, lr=lr,
                         hidden_dim_1=hd_1, hidden_dim_2=hd_2, dropout_rate=drop)
        return res['auc']

    print("Starting Optuna Hyperparameter Search on Fold 1...")
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30) # Runs 30 quick variations

    best_params = study.best_trial.params
    print(f"\nBest Hyperparameters Found: {best_params}\n")
# CV with best parameters
    metrics_history = []
    print("Starting 3D Uni-Mol 2 Optimized Cross-Validation...")

    for i in range(1, 6):
        # Pass the winning parameters into your full 150-epoch training loop!
        fold_metrics = train_fold(fold_idx=i, epochs=150, batch_size=32,
                                  lr=best_params['lr'],
                                  hidden_dim_1=best_params['hidden_dim_1'],
                                  hidden_dim_2=best_params['hidden_dim_2'],
                                  dropout_rate=best_params['dropout_rate'])
        metrics_history.append(fold_metrics)

    print("\nFinal Optimized 5-Fold Cross-Validation Results:")
    for m_key in ["auc", "pr_auc", "f1", "mcc"]:
        values = [m[m_key] for m in metrics_history]
        name = m_key.upper().replace("_", " ")
        print(f"Average {name:7} : {np.mean(values):.4f} ± {np.std(values):.4f}")

[I 2026-07-17 21:24:06,490] A new study created in memory with name: no-name-d65a1fc2-b849-4ed9-b1de-eb57b4469fd3


Starting Optuna Hyperparameter Search on Fold 1...
Starting Fold 1/5
Epoch 001 | Loss: 0.1973 | AUC: 1.0000 | PR-AUC: 1.0000 | F1: 0.9787 | MCC: 0.8756
Epoch 020 | Loss: 0.0570 | AUC: 0.9217 | PR-AUC: 0.9837 | F1: 0.9362 | MCC: 0.6091
Epoch 040 | Loss: 0.0522 | AUC: 0.9130 | PR-AUC: 0.9817 | F1: 0.9388 | MCC: 0.5948


[I 2026-07-17 21:24:11,291] Trial 0 finished with value: 1.0 and parameters: {'hidden_dim_1': 256, 'hidden_dim_2': 64, 'dropout_rate': 0.47194170092475124, 'lr': 0.0036304472351415132}. Best is trial 0 with value: 1.0.


Fold 1 Final -> AUC: 1.0000 | PR-AUC: 1.0000 | F1: 0.9787 | MCC: 0.8756
Starting Fold 1/5
Epoch 001 | Loss: 0.0487 | AUC: 0.8870 | PR-AUC: 0.9707 | F1: 0.9583 | MCC: 0.7430
Epoch 020 | Loss: 0.0770 | AUC: 0.9043 | PR-AUC: 0.9792 | F1: 0.9388 | MCC: 0.5948
Epoch 040 | Loss: 0.0452 | AUC: 0.9391 | PR-AUC: 0.9867 | F1: 0.9583 | MCC: 0.7430


[I 2026-07-17 21:24:16,145] Trial 1 finished with value: 0.9652173913043479 and parameters: {'hidden_dim_1': 256, 'hidden_dim_2': 128, 'dropout_rate': 0.333264395884376, 'lr': 0.0013325356514090145}. Best is trial 0 with value: 1.0.


...

Fold 1 Final -> AUC: 0.9826 | PR-AUC: 0.9963 | F1: 0.9787 | MCC: 0.8756
Starting Fold 1/5
Epoch 001 | Loss: 0.1633 | AUC: 0.8000 | PR-AUC: 0.9539 | F1: 0.9200 | MCC: 0.4128
Epoch 020 | Loss: 0.0640 | AUC: 0.9565 | PR-AUC: 0.9909 | F1: 0.9583 | MCC: 0.7430
Epoch 040 | Loss: 0.0334 | AUC: 0.9130 | PR-AUC: 0.9794 | F1: 0.9583 | MCC: 0.7430


[I 2026-07-17 21:26:27,293] Trial 28 finished with value: 0.982608695652174 and parameters: {'hidden_dim_1': 768, 'hidden_dim_2': 64, 'dropout_rate': 0.3800609883963622, 'lr': 0.0009759213910176528}. Best is trial 0 with value: 1.0.


Fold 1 Final -> AUC: 0.9826 | PR-AUC: 0.9963 | F1: 0.9787 | MCC: 0.8756
Starting Fold 1/5
Epoch 001 | Loss: 0.1412 | AUC: 0.8174 | PR-AUC: 0.9628 | F1: 0.8980 | MCC: 0.2328
Epoch 020 | Loss: 0.0521 | AUC: 0.8609 | PR-AUC: 0.9650 | F1: 0.9583 | MCC: 0.7430
Epoch 040 | Loss: 0.0553 | AUC: 0.9478 | PR-AUC: 0.9891 | F1: 0.9583 | MCC: 0.7430


[I 2026-07-17 21:26:32,088] Trial 29 finished with value: 0.982608695652174 and parameters: {'hidden_dim_1': 512, 'hidden_dim_2': 64, 'dropout_rate': 0.551634830818492, 'lr': 0.006200434600134552}. Best is trial 0 with value: 1.0.


Fold 1 Final -> AUC: 0.9826 | PR-AUC: 0.9963 | F1: 0.9787 | MCC: 0.8756

Best Hyperparameters Found: {'hidden_dim_1': 256, 'hidden_dim_2': 64, 'dropout_rate': 0.47194170092475124, 'lr': 0.0036304472351415132}

Starting 3D Uni-Mol 2 Optimized Cross-Validation...
Starting Fold 1/5
Epoch 001 | Loss: 0.0673 | AUC: 0.9043 | PR-AUC: 0.9801 | F1: 0.9388 | MCC: 0.5948
Epoch 020 | Loss: 0.0469 | AUC: 0.8696 | PR-AUC: 0.9637 | F1: 0.9583 | MCC: 0.7430
Epoch 040 | Loss: 0.0707 | AUC: 0.9217 | PR-AUC: 0.9843 | F1: 0.9048 | MCC: 0.6774
Epoch 060 | Loss: 0.0575 | AUC: 0.8957 | PR-AUC: 0.9779 | F1: 0.8780 | MCC: 0.6255
Epoch 080 | Loss: 0.0496 | AUC: 0.9130 | PR-AUC: 0.9815 | F1: 0.9362 | MCC: 0.6091
Epoch 100 | Loss: 0.0621 | AUC: 0.9043 | PR-AUC: 0.9801 | F1: 0.8780 | MCC: 0.6255
Epoch 120 | Loss: 0.0739 | AUC: 0.9304 | PR-AUC: 0.9849 | F1: 0.9388 | MCC: 0.5948
Epoch 140 | Loss: 0.0928 | AUC: 0.8870 | PR-AUC: 0.9759 | F1: 0.8500 | MCC: 0.5796
Fold 1 Final -> AUC: 0.9739 | PR-AUC: 0.9943 | F1: 0.978

In [11]:
import os
import torch
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, matthews_corrcoef, average_precision_score, f1_score
import warnings
warnings.filterwarnings('ignore')


def train_y_randomization_fold(fold_idx, scrambled_train_labels, epochs=50, batch_size=32, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = UniMolFoldDataset(fold_idx, is_train=True)
    train_dataset.labels = scrambled_train_labels
    val_dataset = UniMolFoldDataset(fold_idx, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = UniMolMLPHead(input_dim=768).to(device)
    criterion = FocalLossWithSmoothing(alpha=0.75, gamma=2.0, smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    NUM_CONFORMERS = 5

    for epoch in range(epochs):
        model.train()
        for embeddings, labels in train_loader:
            if embeddings.size(0) == 1: continue
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(embeddings)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

    # -Evaluate at the end
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for embeddings, labels in val_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            logits = model(embeddings)
            all_logits.extend(logits.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    raw_probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
    raw_labels = np.array(all_labels)

    # Test-Time Conformer Consensus
    if len(raw_probs) % NUM_CONFORMERS == 0:
        probs = raw_probs.reshape(-1, NUM_CONFORMERS).mean(axis=1)
        final_labels = raw_labels.reshape(-1, NUM_CONFORMERS)[:, 0]
    else:
        probs = raw_probs
        final_labels = raw_labels

    preds = (probs > 0.5).astype(int)

    try:
        auc = roc_auc_score(final_labels, probs)
        pr_auc = average_precision_score(final_labels, probs)
        mcc = matthews_corrcoef(final_labels, preds)
        f1 = f1_score(final_labels, preds)
    except ValueError:
        auc, pr_auc, mcc, f1 = 0.5, 0.0, 0.0, 0.0

    # Return the exact state at the end of training
    return {"auc": auc, "mcc": mcc, "pr_auc": pr_auc, "f1": f1}
if __name__ == "__main__":
    N_ITERATIONS = 100
    print(f"Starting 3D Y-Randomization Validation ({N_ITERATIONS} Permutations)...")

    all_rand_aucs = []
    all_rand_mccs = []
    all_rand_prs = []
    all_rand_f1s = []

    for iteration in range(N_ITERATIONS):
        iter_aucs = []
        iter_mccs = []
        iter_prs = []
        iter_f1s = []

        # Run across all 5 folds for this single permutation iteration
        for i in range(1, 6):
            # 1. Load the original train dataset just to get its structure and labels
            temp_dataset = UniMolFoldDataset(i, is_train=True)
            original_labels = temp_dataset.labels

            # 2. Extract molecule-level labels (Assuming exactly 5 conformers per molecule)
            num_mols = len(original_labels) // 5
            mol_labels = original_labels.view(num_mols, 5)[:, 0]

            # 3. Shuffle labels at the molecule level
            perm = torch.randperm(num_mols)
            shuffled_mol_labels = mol_labels[perm]

            # 4. Broadcast the shuffled labels back to the 5 conformers
            scrambled_conformer_labels = shuffled_mol_labels.unsqueeze(1).repeat(1, 5).view(-1)

            # 5. Train the fold with these explicitly scrambled labels
            res = train_y_randomization_fold(fold_idx=i, scrambled_train_labels=scrambled_conformer_labels, epochs=50)
            iter_aucs.append(res['auc'])
            iter_mccs.append(res['mcc'])
            iter_prs.append(res['pr_auc'])
            iter_f1s.append(res['f1'])

        # Calculate means for this permutation iteration
        mean_iter_auc = np.mean(iter_aucs)
        mean_iter_mcc = np.mean(iter_mccs)
        mean_iter_pr = np.mean(iter_prs)
        mean_iter_f1 = np.mean(iter_f1s)

        all_rand_aucs.append(mean_iter_auc)
        all_rand_mccs.append(mean_iter_mcc)
        all_rand_prs.append(mean_iter_pr)
        all_rand_f1s.append(mean_iter_f1)

        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f"Run {iteration+1}/{N_ITERATIONS} Mean Metrics | AUC: {mean_iter_auc:.4f} | PR-AUC: {mean_iter_pr:.4f} | F1: {mean_iter_f1:.4f} | MCC: {mean_iter_mcc:.4f}")

    print("\nY-Randomization Final Results: ")
    print(f"Average Random AUC : {np.mean(all_rand_aucs):.4f} ± {np.std(all_rand_aucs):.4f}")
    print(f"Average Random PR-AUC: {np.mean(all_rand_prs):.4f} ± {np.std(all_rand_prs):.4f}")
    print(f"Average Random F1  : {np.mean(all_rand_f1s):.4f} ± {np.std(all_rand_f1s):.4f}")
    print(f"Average Random MCC : {np.mean(all_rand_mccs):.4f} ± {np.std(all_rand_mccs):.4f}")

Starting 3D Y-Randomization Validation (100 Permutations)...
Run 1/100 Mean Metrics | AUC: 0.4550 | PR-AUC: 0.6491 | F1: 0.6639 | MCC: -0.0543
Run 10/100 Mean Metrics | AUC: 0.4616 | PR-AUC: 0.6278 | F1: 0.6407 | MCC: -0.0409
Run 20/100 Mean Metrics | AUC: 0.5496 | PR-AUC: 0.6680 | F1: 0.6247 | MCC: 0.0056
Run 30/100 Mean Metrics | AUC: 0.5725 | PR-AUC: 0.6781 | F1: 0.7284 | MCC: 0.1097
Run 40/100 Mean Metrics | AUC: 0.4513 | PR-AUC: 0.6282 | F1: 0.6051 | MCC: -0.1463
Run 50/100 Mean Metrics | AUC: 0.5178 | PR-AUC: 0.6272 | F1: 0.6709 | MCC: 0.0362
Run 60/100 Mean Metrics | AUC: 0.4009 | PR-AUC: 0.6164 | F1: 0.6323 | MCC: -0.1175
Run 70/100 Mean Metrics | AUC: 0.5200 | PR-AUC: 0.7007 | F1: 0.6892 | MCC: -0.0548
Run 80/100 Mean Metrics | AUC: 0.4315 | PR-AUC: 0.6117 | F1: 0.6561 | MCC: -0.1017
Run 90/100 Mean Metrics | AUC: 0.4673 | PR-AUC: 0.6071 | F1: 0.5514 | MCC: -0.0594
Run 100/100 Mean Metrics | AUC: 0.4912 | PR-AUC: 0.6660 | F1: 0.6826 | MCC: 0.1319

Y-Randomization Final Results

In [12]:
import os
import torch
import numpy as np
import pandas as pd
from rdkit import Chem
from unimol_tools import UniMolRepr


BLIND_SET_SDF = "BlindSet_3D.sdf"
FOLDS = 5
NUM_CONFORMERS = 5

def run_ensemble_inference(sdf_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Extract Embeddings for Blind Set
    print("Loading Uni-Mol and extracting blind set representations...")
    unimol = UniMolRepr(data_type='molecule', remove_hs=False, model_name='unimolv2')
    supplier = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=True)

    custom_data = {"atoms": [], "coordinates": []}
    parent_ids = []

    # Track IDs to ensure we group the 5 conformers correctly
    for idx, mol in enumerate(supplier):
        if mol is None: continue

        # Grab the parent ID so all 5 conformers share the exact same string
        mol_id = mol.GetProp("Parent_ID") if mol.HasProp("Parent_ID") else f"Unknown_{idx//NUM_CONFORMERS}"
        parent_ids.append(mol_id)

        conf = mol.GetConformer()
        atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
        coords = conf.GetPositions().tolist()

        custom_data["atoms"].append(atoms)
        custom_data["coordinates"].append(coords)

    repr_output = unimol.get_repr(custom_data, return_atomic_reprs=False)

    # Handle UniMol dictionary output
    if isinstance(repr_output, dict):
        cls_embeddings = np.array(repr_output['cls_repr'])
    elif isinstance(repr_output, list) or isinstance(repr_output, np.ndarray):
        if len(repr_output) > 0 and isinstance(repr_output[0], dict):
            cls_embeddings = np.array([item['cls_repr'] for item in repr_output])
        else:
            cls_embeddings = np.array(repr_output)
    else:
        raise ValueError(f"Unexpected output type from Uni-Mol: {type(repr_output)}")

    blind_embeddings = torch.tensor(cls_embeddings, dtype=torch.float32).to(device)

    # 2. Load the 5 Trained Models
    models = []
    for i in range(1, FOLDS + 1):
        # Ensure hidden_dims match the best params found Optuna search
        model = UniMolMLPHead(input_dim=768, hidden_dim_1=256, hidden_dim_2=64).to(device)
        model.load_state_dict(torch.load(f"best_model_fold_{i}.pth", map_location=device))
        model.eval()
        models.append(model)

    # 3. Ensemble Prediction
    print("Running 5-Model Ensemble Inference...")
    all_model_probs = []

    with torch.no_grad():
        for model in models:
            logits = model(blind_embeddings)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_model_probs.append(probs)

    # Average probabilities across the 5 models
    ensemble_conformer_probs = np.mean(all_model_probs, axis=0)

    # 4. Test-Time Conformer Consensus (Average by actual ID)
    print("Applying Test-Time Conformer Consensus...")

    # Flatten probabilities to 1D just in case
    ensemble_conformer_probs = ensemble_conformer_probs.flatten()

    if len(ensemble_conformer_probs) != len(parent_ids):
        print("ERROR: Probability count does not match ID count!")
        return

    # Create a DataFrame of all individual conformer predictions
    df_conformers = pd.DataFrame({
        "ID": parent_ids,
        "Conf_Prob": ensemble_conformer_probs
    })

    # Group by the parent ID and calculate the mean probability across however many conformers survived
    df_grouped = df_conformers.groupby("ID", as_index=False)["Conf_Prob"].mean()

    # 5. Output Final Results
    print("\nBlind Set Predictions")
    results = []

    for _, row in df_grouped.iterrows():
        mol_id = row["ID"]
        prob = row["Conf_Prob"]
        prediction = "ACTIVE" if prob >= 0.5 else "INACTIVE"

        print(f"ID: {mol_id:15} | Probability: {prob:.4f} | Prediction: {prediction}")
        results.append({
            "ID": mol_id,
            "Probability": prob,
            "Prediction": prediction
        })

    # Save to CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv("BlindSet_UniMol_Predictions.csv", index=False)
    print("\nPredictions saved to BlindSet_UniMol_Predictions.csv")


if __name__ == "__main__":
    if os.path.exists(BLIND_SET_SDF):
        run_ensemble_inference(BLIND_SET_SDF)
    else:
        print(f"Blind set SDF not found at {BLIND_SET_SDF}.")

Loading Uni-Mol and extracting blind set representations...


2026-07-17 21:52:50 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/modelzoo/84M/checkpoint.pt
INFO:Uni-Mol Tools:Loading pretrained weights from /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/modelzoo/84M/checkpoint.pt
2026-07-17 21:52:50 | unimol_tools/tasks/trainer.py | 78 | INFO | Uni-Mol Tools | Number of GPUs available: 1
INFO:Uni-Mol Tools:Number of GPUs available: 1
2026-07-17 21:52:50 | unimol_tools/tasks/trainer.py | 98 | INFO | Uni-Mol Tools | Using single GPU.
INFO:Uni-Mol Tools:Using single GPU.
100%|██████████| 10/10 [00:03<00:00,  2.53it/s]


Running 5-Model Ensemble Inference...
Applying Test-Time Conformer Consensus...

Blind Set Predictions
ID: 10              | Probability: 0.7345 | Prediction: ACTIVE
ID: 104             | Probability: 0.6955 | Prediction: ACTIVE
ID: 107             | Probability: 0.9912 | Prediction: ACTIVE
ID: 108             | Probability: 0.9944 | Prediction: ACTIVE
ID: 111             | Probability: 0.9910 | Prediction: ACTIVE
ID: 112             | Probability: 0.9910 | Prediction: ACTIVE
ID: 113             | Probability: 0.9927 | Prediction: ACTIVE
ID: 115             | Probability: 0.9952 | Prediction: ACTIVE
ID: 116             | Probability: 0.6808 | Prediction: ACTIVE
ID: 118             | Probability: 0.9506 | Prediction: ACTIVE
ID: 119             | Probability: 0.9511 | Prediction: ACTIVE
ID: 120             | Probability: 0.9938 | Prediction: ACTIVE
ID: 122             | Probability: 0.9971 | Prediction: ACTIVE
ID: 123             | Probability: 0.8142 | Prediction: ACTIVE
ID: 124        